# Week 10 – LLM‑Guided Backtest

This notebook backtests the LLM‑guided strategy using the historical simulation from Week 8.
It runs the agent for each day in a historical window, collects the daily signals and the realized P&L from closed trades,
then computes performance metrics (Sharpe ratio, max drawdown, win rate) using the functions in `src/evaluation/metrics.py`.


In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('.')  # allow import from src
from src.evaluation.metrics import compute_sharpe, compute_max_drawdown, compute_win_rate
from src.simulation.orchestrator import simulate_historical


In [ ]:
# Run the historical simulation for a single ticker (e.g. AAPL) over a chosen window.
# You can adjust the ticker, start_date, end_date, and starting capital as needed.

ticker = "AAPL"
start_date = "2023-01-01"
end_date   = "2023-12-31"
starting_capital = 10_000.0

print(f"Running simulation for {ticker} from {start_date} to {end_date}...")
result = simulate_historical(
    ticker=ticker,
    start_date=start_date,
    end_date=end_date,
    portfolio_value=starting_capital,
    verbose=False  # set True to see per‑day output
)


In [ ]:
# Extract the list of closed trades from the simulation.
closed_trades = result.get('closed_trades', [])
print(f"Number of closed trades: {len(closed_trades)}")

# If there are closed trades, compute a simple equity curve from realized P&L.
if closed_trades:
    # Each trade dict should have a 'pnl' field (profit/loss in currency).
    pnl_list = [t.get('pnl', 0.0) for t in closed_trades]
    equity = [starting_capital]
    for pnl in pnl_list:
        equity.append(equity[-1] + pnl)
    equity = np.array(equity)
    # Daily returns from the equity curve (for Sharpe/drawdown).
    returns = np.diff(equity) / equity[:-1]
    # Compute metrics using our module.
    sharpe = compute_sharpe(returns)
    max_dd = compute_max_drawdown(equity)
    # Approximate win rate from closed trades.
    wins = [t for t in closed_trades if t.get('pnl', 0.0) > 0]
    win_rate = len(wins) / len(closed_trades) if closed_trades else 0.0
    print('\n=== LLM‑Guided Strategy Performance ===')
    print(f'Sharpe Ratio: {sharpe:.3f}')
    print(f'Max Drawdown: {max_dd:.2%}')
    print(f'Win Rate:   {win_rate:.2%}')
    print(f'Total Return: {(equity[-1]/starting_capital - 1):.2%}')
    print(f'Final Equity: ${equity[-1]:,.2f}')
else:
    print('No closed trades in this simulation window.')
    sharpe = None
    max_dd = None
    win_rate = None
    equity = np.array([starting_capital])


In [ ]:
# Plot the equity curve (realized P&L only).
plt.figure(figsize=(12,6))
# x‑axis: we don't have a date per trade, so just trade index.
trade_idx = np.arange(len(equity))
plt.plot(trade_idx, equity, marker='o', linestyle='-')
plt.title(f'Equity Curve – LLM‑Guided Strategy ({ticker})')
plt.ylabel('Portfolio Value ($)')
plt.xlabel('Trade # (closed trades only)')
plt.grid(True)
plt.show()


In [ ]:
# OPTIONAL: Extract daily signals from the simulation.
# The `result['daily_results']` contains a list of dicts, each with a 'date' and 'result'.
# The 'result' is the output of `run_daily_cycle`; we assume it includes a 'signal' dict.
# We'll try to pull the signal fields if they exist.

daily_results = result.get('daily_results', [])
signal_dates = []
signal_actions = []
signal_confidence = []
for dr in daily_results:
    if 'result' in dr and isinstance(dr['result'], dict):
        res = dr['result']
        # Adjust the keys below if your run_daily_cycle returns different names.
signal = res.get('signal', {})
        if signal:
            signal_dates.append(dr['date'])
            signal_actions.append(signal.get('action', None))
            signal_confidence.append(signal.get('confidence', None))

if signal_dates:
    print(f'Extracted {len(signal_dates)} daily signals.')
    # Show first few as a DataFrame.
    signals_df = pd.DataFrame({
        'date': signal_dates,
        'action': signal_actions,
        'confidence': signal_confidence
    })
    print(signals_df.head(10))
else:
    print('No signal information found in daily_results.')


In [ ]:
# OPTIONAL: Run a quick comparison with the buy‑and‑hold baseline for the same period.
# We'll compute buy‑and‑hold equity curve using the same price data the simulation used.
# This requires loading the indicator CSV (or just close prices).
# For simplicity, we skip this here; you can add it in the Thursday notebook.
print('\nTip: To compare with baseline, run the Thursday notebook or add buy‑and‑hold calculations here.')
